# grahspj: DR16Q Photometry With Bandwagon and SVI

This notebook demonstrates a bulk quasar workflow:

- download the SDSS DR16Q value-added catalog from the Illinois quasar archive
- draw a reproducible quasar sample
- query GALEX, SDSS, and AllWISE photometry with `bandwagon` through CDS XMatch
- convert matched magnitudes and errors into long-form flux-density photometry
- fit each quasar SED with the `grahspj` SVI/MAP path

The default fit settings are intentionally light enough for a demo. Increase `SVI_STEPS` and rerun the fitting cells for production-quality MAP solutions.

`fit_map()` now uses staged continuum-first initialization by default, matching the standard `fit(..., fit_method="optax")` path. Set `staged=False` in direct `fit_map()` calls only when comparing to the older single-stage optimizer.


## Prerequisites

This notebook assumes:

- you are running from `grahspj/notebooks/`
- `grahspj` dependencies are installed
- the sibling `bandwagon` package exists at `../../bandwagon/src`, or `bandwagon` is installed in your environment
- internet access is available for the catalog download and CDS XMatch calls
- a valid DSPS SSP file is available; by default this notebook looks for `../../jaxqsofit/tempdata.h5`

The DR16Q catalog URL used here is:

`http://quasar.astro.illinois.edu/paper_data/DR16Q/dr16q_prop_May01_2024.fits.gz`


In [ ]:
from pathlib import Path
import sys

import astropy.units as u
import matplotlib.pyplot as plt
import numpy as np
from astropy.coordinates import SkyCoord
from astropy.table import Table
from astropy.utils.data import download_file
from tqdm.auto import tqdm

project_root = Path.cwd().resolve().parent
grahspj_src = project_root / "src"
bandwagon_src = project_root.parent / "bandwagon" / "src"
for path in (grahspj_src, bandwagon_src):
    if path.is_dir() and str(path) not in sys.path:
        sys.path.insert(0, str(path))

from bandwagon import DEFAULT_CATALOGS, matches_to_photometry, xmatch_catalogs
from grahspj.config import AGNConfig, FilterSet, FitConfig, GalaxyConfig, InferenceConfig, LikelihoodConfig, Observation, PhotometryData
from grahspj.core import GRAHSPJ
from grahspj.mplstyle import style_path

plt.style.use(style_path())


In [ ]:
DR16Q_URL = "http://quasar.astro.illinois.edu/paper_data/DR16Q/dr16q_prop_May01_2024.fits.gz"
LOCAL_DR16Q_PATH = project_root.parent / "qvc" / "data" / "dr16q_prop_May01_2024.fits"
SAMPLE_SIZE = 1
RANDOM_SEED = 20260419
SVI_STEPS = 600
SVI_LEARNING_RATE = 5e-3
MIN_BANDS_TO_FIT = 5

# Set this lower while iterating locally, or keep it at SAMPLE_SIZE for the full demo.
N_FIT = SAMPLE_SIZE

# Keep noisier W3/W4 points in the demo while still dropping extreme catalog rows.
MAX_MAG_ERR = 1.0

FILTER_ORDER = {
    "FUV_galex": 0,
    "NUV_galex": 1,
    "u_sdss": 2,
    "g_sdss": 3,
    "r_sdss": 4,
    "i_sdss": 5,
    "z_sdss": 6,
    "W1": 7,
    "W2": 8,
    "W3": 9,
    "W4": 10,
}

# CDS XMatch defaults cover GALEX FUV/NUV, SDSS ugriz, and AllWISE W1-W4.
DEFAULT_CATALOGS


## Download DR16Q


In [ ]:
if LOCAL_DR16Q_PATH.is_file():
    catalog_path = LOCAL_DR16Q_PATH
else:
    catalog_path = Path(download_file(DR16Q_URL, cache=True, show_progress=True))

dr16q = Table.read(catalog_path)

print(f"Downloaded/read {len(dr16q):,} DR16Q rows")
print("First 25 columns:", dr16q.colnames[:25])
dr16q[:3]


In [ ]:
ra = np.asarray(dr16q["RA"], dtype=float)
dec = np.asarray(dr16q["DEC"], dtype=float)
redshift = np.asarray(dr16q["Z_DR16Q"], dtype=float)
valid = np.isfinite(ra) & np.isfinite(dec) & np.isfinite(redshift) & (redshift > 0.0)

rng = np.random.default_rng(RANDOM_SEED)
valid_idx = np.flatnonzero(valid)
sample_idx = rng.choice(valid_idx, size=min(SAMPLE_SIZE, valid_idx.size), replace=False)
sample = dr16q[sample_idx]

raw_names = np.asarray([str(value).strip() for value in sample["SDSS_NAME"]], dtype=object)

# Use guaranteed-unique local IDs for grouping XMatch results. Catalog names are
# kept as metadata because some table columns can be blank or non-unique.
source_ids = np.asarray([f"DR16Q_{int(i)}" for i in sample_idx], dtype=object)

coords = SkyCoord(
    ra=np.asarray(sample["RA"], dtype=float) * u.deg,
    dec=np.asarray(sample["DEC"], dtype=float) * u.deg,
    frame="icrs",
)

sample_meta = {
    str(source_id): {
        "source_id": str(source_id),
        "catalog_name": str(raw_name),
        "row_index": int(row_index),
        "ra": float(row["RA"]),
        "dec": float(row["DEC"]),
        "redshift": float(row["Z_DR16Q"]),
    }
    for source_id, raw_name, row_index, row in zip(source_ids, raw_names, sample_idx, sample)
}

print("Using columns: ra='RA', dec='DEC', redshift='Z_DR16Q', id='SDSS_NAME'")
print(f"Sample size: {len(sample):,}")
print("First source IDs:", source_ids[:5].tolist())


## Query Photometry With Bandwagon


In [ ]:
# This cell performs three CDS XMatch requests: GALEX AIS, SDSS DR16, and AllWISE.
# If you are rerunning repeatedly, consider writing `photometry` to ECSV after the first successful run.
matches = xmatch_catalogs(coords, source_id=source_ids)

for key, table in matches.items():
    print(f"{key:10s}: {len(table):6d} matched rows; columns={table.colnames[:12]}")


In [ ]:
photometry = matches_to_photometry(matches, max_mag_err=MAX_MAG_ERR)
print(f"Long-form photometry rows: {len(photometry):,}")

if len(photometry):
    filters, filter_counts = np.unique(np.asarray(photometry["filter_name"], dtype=str), return_counts=True)
    print("Rows by filter:")
    for name, count in zip(filters, filter_counts):
        print(f"  {name:10s} {count:5d}")

photometry[:12]


In [ ]:
# Keep only sources with enough usable photometric bands for a constrained demo fit.
# Prefer examples with W4, then W3, then the largest number of bands so the
# notebook's displayed fit exercises the long-wavelength WISE bands when present.
phot_source_ids = np.asarray(photometry["source_id"], dtype=str) if len(photometry) else np.asarray([], dtype=str)
unique_ids, counts = np.unique(phot_source_ids, return_counts=True)
source_band_sets = {
    sid: set(np.asarray(photometry[phot_source_ids == sid]["filter_name"], dtype=str))
    for sid in unique_ids
}

def source_priority(source_id):
    bands = source_band_sets[str(source_id)]
    return (
        "W4" in bands,
        "W3" in bands,
        len(bands),
    )

fit_source_ids = [sid for sid, count in zip(unique_ids, counts) if count >= MIN_BANDS_TO_FIT]
fit_source_ids = sorted(fit_source_ids, key=source_priority, reverse=True)[:N_FIT]

print(f"Sources with >= {MIN_BANDS_TO_FIT} bands: {len(fit_source_ids):,}")
print(f"Will fit: {len(fit_source_ids):,}")
if fit_source_ids:
    first_bands = sorted(source_band_sets[fit_source_ids[0]], key=lambda name: FILTER_ORDER.get(name, 999))
    print("First fit source bands:", first_bands)

plt.figure(figsize=(7, 4))
plt.hist(counts, bins=np.arange(0.5, max(counts, default=0) + 1.5, 1.0), color="0.25")
plt.xlabel("usable photometric bands per source")
plt.ylabel("number of sources")
plt.tight_layout()


## Build `grahspj` Configurations


In [ ]:
dsps_ssp_fn = project_root.parent / "jaxqsofit" / "tempdata.h5"
assert dsps_ssp_fn.is_file(), f"DSPS SSP file not found: {dsps_ssp_fn}"


def rows_for_source(source_id):
    mask = np.asarray(photometry["source_id"], dtype=str) == str(source_id)
    rows = photometry[mask]
    order = np.argsort([FILTER_ORDER.get(str(name), 999) for name in rows["filter_name"]])
    return rows[order]


def build_fit_config(source_id, rows):
    meta = sample_meta[str(source_id)]
    filter_names = [str(name) for name in rows["filter_name"]]
    fluxes = np.asarray(rows["flux_mjy"], dtype=float)
    errors = np.asarray(rows["flux_err_mjy"], dtype=float)

    # Avoid unrealistically tiny catalog errors in a heterogeneous multi-epoch SED.
    errors = np.maximum(errors, 0.03 * fluxes)

    return FitConfig(
        observation=Observation(
            object_id=str(source_id),
            redshift=float(meta["redshift"]),
            fit_redshift=False,
            ra=float(meta["ra"]),
            dec=float(meta["dec"]),
        ),
        photometry=PhotometryData(
            filter_names=filter_names,
            fluxes=fluxes.tolist(),
            errors=errors.tolist(),
            is_upper_limit=[False] * len(rows),
            psf_fwhm_arcsec=[float(value) for value in rows["psf_fwhm_arcsec"]],
        ),
        filters=FilterSet(
            speclite_names={str(row["filter_name"]): str(row["speclite_name"]) for row in rows},
            use_grahsp_database=False,
        ),
        galaxy=GalaxyConfig(
            dsps_ssp_fn=str(dsps_ssp_fn),
            n_wave=768,
        ),
        agn=AGNConfig(agn_type=1),
        likelihood=LikelihoodConfig(
            systematics_width=0.08,
            variability_uncertainty=True,
            use_host_capture_model=True,
        ),
        inference=InferenceConfig(
            map_steps=SVI_STEPS,
            learning_rate=SVI_LEARNING_RATE,
            seed=RANDOM_SEED,
        ),
        prior_config={
            "log_stellar_mass": {"loc": 10.5, "scale": 1.25},
            "fracAGN_5100": {"loc": 0.75, "scale": 0.2},
            "ebv_gal": {"scale": 0.2},
            "ebv_agn": {"scale": 0.2},
        },
    )

example_rows = rows_for_source(fit_source_ids[0]) if fit_source_ids else None
example_cfg = build_fit_config(fit_source_ids[0], example_rows) if fit_source_ids else None
print(example_cfg)


## Fit With SVI

`GRAHSPJ.fit_map()` uses NumPyro `SVI` with an `AutoDelta` guide, so the result is a MAP estimate. The loop below runs that SVI path for every selected source and records a compact summary.


In [ ]:
fit_summaries = []
failed_fits = []
example_fitter = None

for n, source_id in enumerate(tqdm(fit_source_ids, desc="SVI fits")):
    rows = rows_for_source(source_id)
    cfg = build_fit_config(source_id, rows)
    fitter = GRAHSPJ(cfg)
    try:
        # Direct fit_map uses staged continuum-first initialization by default.
        map_result = fitter.fit_map(
            steps=SVI_STEPS,
            learning_rate=SVI_LEARNING_RATE,
            progress_bar=False,
        )
        summary = fitter.summary()
        loss = np.asarray(map_result.get("losses", []), dtype=float)
        fit_summaries.append(
            {
                "source_id": str(source_id),
                "redshift": float(cfg.observation.redshift),
                "n_bands": len(rows),
                "final_loss": float(loss[-1]) if loss.size else np.nan,
                "log_stellar_mass_fit": summary.get("log_stellar_mass_fit", np.nan),
                "fracAGN_5100_median": summary.get("fracAGN_5100_median", np.nan),
                "log_agn_bol_luminosity_median": summary.get("log_agn_bol_luminosity_median", np.nan),
            }
        )
        if example_fitter is None:
            example_fitter = fitter
    except Exception as exc:
        failed_fits.append({"source_id": str(source_id), "error": repr(exc)})

fit_summary = Table(rows=fit_summaries)
print(f"Successful fits: {len(fit_summary):,}")
print(f"Failed fits: {len(failed_fits):,}")
fit_summary[:10]


In [ ]:
if failed_fits:
    Table(rows=failed_fits)[:10]
else:
    print("No failed fits.")


## Inspect One Fit


In [ ]:
if example_fitter is None:
    raise RuntimeError("No successful fit to plot.")

pred = example_fitter.predict()
fig = example_fitter.plot_sed(show=True)


In [ ]:
phot_wave = np.asarray([flt.effective_wavelength for flt in example_fitter.context.filters], dtype=float)
model_flux = np.asarray(pred["pred_fluxes"][0], dtype=float)

print("filter, eff_wave_A, obs_mJy, err_mJy, model_mJy")
for name, wave, obs, err, model in zip(
    example_fitter.config.photometry.filter_names,
    phot_wave,
    example_fitter.config.photometry.fluxes,
    example_fitter.config.photometry.errors,
    model_flux,
):
    print((name, wave, obs, err, model))


## Save Optional Outputs

Uncomment these lines if you want local reusable artifacts from the demo run.


In [ ]:
# output_dir = project_root / "notebook_outputs" / "05_dr16q_bandwagon_svi"
# output_dir.mkdir(parents=True, exist_ok=True)
# photometry.write(output_dir / "dr16q_sample_bandwagon_photometry.ecsv", overwrite=True)
# fit_summary.write(output_dir / "dr16q_sample_svi_summary.ecsv", overwrite=True)


## Notes

- GALEX and SDSS magnitudes are treated as AB magnitudes.
- AllWISE magnitudes are converted from Vega magnitudes with the standard WISE zero-points used by Bandwagon.
- This demo uses fixed catalog redshifts from DR16Q. To fit redshift, set `fit_redshift=True` and provide an uncertainty or redshift prior in the `FitConfig`.
- The raw XMatch tables are still available in `matches` if you need catalog quality flags beyond the normalized photometry table.
